# 02. Tiền xử lý dữ liệu (Data Preprocessing)

**Đề tài:** Hệ thống khuyến nghị cây trồng

**Người thực hiện:** Tuấn Hùng

## 1. Giới thiệu

Notebook này thực hiện tiền xử lý bộ dữ liệu Crop Recommendation nhằm chuẩn bị dữ liệu cho các thuật toán học máy.

Các bước làm sạch và chia train/test được triển khai trong `src/preprocessing.py`. Notebook gọi lại các hàm này để trình bày và kiểm tra kết quả.

Các công việc chính:

- Đọc và kiểm tra dữ liệu.
- Kiểm tra giá trị thiếu, trùng lặp và không hợp lệ.
- Xác định các đặc trưng đầu vào và nhãn mục tiêu.
- Chia dữ liệu thành tập huấn luyện và kiểm tra.
- Lưu dữ liệu đã xử lý để bàn giao cho nhóm huấn luyện mô hình.

In [ ]:
# Nhập các thư viện cần thiết
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Hỗ trợ chạy notebook từ thư mục gốc hoặc thư mục notebooks/
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.preprocessing import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    clean_data,
    split_data,
)

features = list(FEATURE_COLUMNS)

# Cấu hình tiếng Việt cho biểu đồ
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

# Đọc dữ liệu từ đường dẫn thống nhất trong project
data_path = project_dir / "data" / "raw" / "Crop_recommendation.csv"
df = pd.read_csv(data_path)

print("Đọc dữ liệu thành công!")
print("Kích thước dữ liệu:", df.shape)

display(df.head())

## 3. Kiểm tra chất lượng dữ liệu

Kiểm tra giá trị thiếu, bản ghi trùng lặp và kiểu dữ liệu nhằm xác định những vấn đề cần xử lý trước khi huấn luyện mô hình.

In [2]:
# Kiểm tra giá trị thiếu
print("Số lượng giá trị thiếu:")
print(df.isnull().sum())

# Kiểm tra dữ liệu trùng lặp
print("\nSố dòng trùng lặp:")
print(df.duplicated().sum())

# Kiểm tra kiểu dữ liệu
print("\nKiểu dữ liệu của các thuộc tính:")
print(df.dtypes)

# Kiểm tra số lớp nhãn
print("\nSố loại cây trồng:")
print(df[TARGET_COLUMN].nunique())

Số lượng giá trị thiếu:
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

Số dòng trùng lặp:
0

Kiểu dữ liệu của các thuộc tính:
N                int64
P                int64
K                int64
temperature    float64
humidity       float64
ph             float64
rainfall       float64
label              str
dtype: object

Số loại cây trồng:
22


## 4. Làm sạch và kiểm tra dữ liệu hợp lệ

Mục tiêu của bước này là đảm bảo dữ liệu đáp ứng các điều kiện cần thiết trước khi xây dựng mô hình học máy.

Các công việc thực hiện:
- Loại bỏ các dòng dữ liệu trùng lặp.
- Kiểm tra và xử lý giá trị bị thiếu.
- Kiểm tra kiểu dữ liệu của các đặc trưng.
- Phát hiện các giá trị nằm ngoài phạm vi hợp lệ.
- Đảm bảo dữ liệu đầu ra có chất lượng phù hợp để huấn luyện mô hình.

Các quy tắc trên được thực hiện bởi hàm `clean_data(df)` trong `src/preprocessing.py`.

In [3]:
# Gọi hàm làm sạch dùng chung; dữ liệu gốc df không bị thay đổi
df_clean = clean_data(df)

print("Số dòng bị loại bỏ:", len(df) - len(df_clean))
print("Số mẫu ban đầu:", len(df))
print("Số mẫu sau khi làm sạch:", len(df_clean))
print("Số loại cây trồng:", df_clean[TARGET_COLUMN].nunique())

Số dòng bị loại bỏ: 0
Số mẫu ban đầu: 2200
Số mẫu sau khi làm sạch: 2200
Số loại cây trồng: 22


## 5. Xác định đặc trưng đầu vào và nhãn mục tiêu

Để xây dựng mô hình phân loại cây trồng, dữ liệu được chia thành:

- X: Gồm 7 đặc trưng nông nghiệp và khí hậu.
- y: Nhãn loại cây trồng cần dự đoán.

Việc phân chia này giúp tách biệt dữ liệu đầu vào và kết quả mong muốn của mô hình.

In [4]:
# Tách đặc trưng đầu vào
X = df_clean[features]

# Tách nhãn mục tiêu
y = df_clean[TARGET_COLUMN]

print("Kích thước tập đặc trưng X:", X.shape)
print("Kích thước tập nhãn y:", y.shape)

print("\n5 dòng đặc trưng đầu vào:")
display(X.head())

print("\n5 nhãn mục tiêu đầu tiên:")
display(y.head())

Kích thước tập đặc trưng X: (2200, 7)
Kích thước tập nhãn y: (2200,)

5 dòng đặc trưng đầu vào:


,N,P,K,temperature,humidity,ph,rainfall
0,90,42,43,20.879744,82.002744,6.502985,202.935536
1,85,58,41,21.770462,80.319644,7.038096,226.655537
2,60,55,44,23.004459,82.320763,7.840207,263.964248
3,74,35,40,26.491096,80.158363,6.980401,242.864034
4,78,42,42,20.130175,81.604873,7.628473,262.717340



5 nhãn mục tiêu đầu tiên:


0    rice
1    rice
2    rice
3    rice
4    rice
Name: label, dtype: str

## 6. Chia dữ liệu thành tập huấn luyện và kiểm tra

Bộ dữ liệu được chia theo tỷ lệ 80/20:

- 80% dữ liệu dùng để huấn luyện mô hình.
- 20% dữ liệu dùng để kiểm tra mô hình.

Sử dụng phương pháp phân chia có phân tầng (stratify) để duy trì tỷ lệ các loại cây trồng giữa hai tập.

Tham số random_state=42 giúp kết quả phân chia có thể được tái lập.

Sử dụng hàm `split_data(X, y, test_size=0.2, random_state=42)` từ `src.preprocessing`.

In [5]:
# Chia dữ liệu bằng hàm dùng chung, có phân tầng theo nhãn
X_train, X_test, y_train, y_test = split_data(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("KẾT QUẢ PHÂN CHIA DỮ LIỆU")
print("-" * 45)

print("Số mẫu huấn luyện:", len(X_train))
print("Số mẫu kiểm tra:", len(X_test))

print("\nKích thước X_train:", X_train.shape)
print("Kích thước X_test:", X_test.shape)
print("Kích thước y_train:", y_train.shape)
print("Kích thước y_test:", y_test.shape)

print("\nPhân bố nhãn trong tập Train:")
print(y_train.value_counts().sort_index())

print("\nPhân bố nhãn trong tập Test:")
print(y_test.value_counts().sort_index())

KẾT QUẢ PHÂN CHIA DỮ LIỆU
---------------------------------------------
Số mẫu huấn luyện: 1760
Số mẫu kiểm tra: 440

Kích thước X_train: (1760, 7)
Kích thước X_test: (440, 7)
Kích thước y_train: (1760,)
Kích thước y_test: (440,)

Phân bố nhãn trong tập Train:
label
apple          80
banana         80
blackgram      80
chickpea       80
coconut        80
coffee         80
cotton         80
grapes         80
jute           80
kidneybeans    80
lentil         80
maize          80
mango          80
mothbeans      80
mungbean       80
muskmelon      80
orange         80
papaya         80
pigeonpeas     80
pomegranate    80
rice           80
watermelon     80
Name: count, dtype: int64

Phân bố nhãn trong tập Test:
label
apple          20
banana         20
blackgram      20
chickpea       20
coconut        20
coffee         20
cotton         20
grapes         20
jute           20
kidneybeans    20
lentil         20
maize          20
mango          20
mothbeans      20
mungbean       20
muskm

## 7. Lưu dữ liệu đã tiền xử lý

Sau khi làm sạch và phân chia dữ liệu, các tập dữ liệu được lưu lại để phục vụ giai đoạn huấn luyện mô hình.

Các sản phẩm đầu ra bao gồm:

- Dữ liệu Train và Test giữ nguyên thang đo: `X_train.csv` và `X_test.csv`.
- Nhãn mục tiêu tương ứng: `y_train.csv` và `y_test.csv`.

Việc lưu riêng các tập dữ liệu giúp đảm bảo tính nhất quán và khả năng tái lập kết quả huấn luyện.

Notebook 03 thực hiện chuẩn hóa bằng `StandardScaler` trong `Pipeline` của SoftMax, KNN và SVM. Scaler được fit riêng trên phần train của từng fold khi chạy cross-validation để tránh rò rỉ dữ liệu. Các mô hình còn lại dùng dữ liệu gốc.


In [ ]:
# Tạo thư mục lưu dữ liệu
output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Lưu dữ liệu giữ nguyên thang đo
X_train.to_csv(output_dir / "X_train.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)

# Lưu nhãn mục tiêu
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

print("ĐÃ LƯU DỮ LIỆU THÀNH CÔNG!")
print("\nThư mục lưu:", output_dir)

print("\nDanh sách file đã tạo:")

for filename in ("X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"):
    print("-", filename)


In [ ]:
print("KIỂM TRA KẾT QUẢ TIỀN XỬ LÝ")
print("-" * 50)

print("Số mẫu Train:", len(X_train))
print("Số mẫu Test:", len(X_test))

print("\nSố đặc trưng:", X_train.shape[1])

print("\nGiá trị thiếu trong Train:")
print(X_train.isnull().sum().sum())

print("\nGiá trị thiếu trong Test:")
print(X_test.isnull().sum().sum())

# Xác nhận các tập dữ liệu tương ứng với nhãn
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

# Xác nhận không còn giá trị vô hạn hoặc NaN
assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_test.to_numpy()).all()

print("\nKiểm tra hoàn tất!")


## 8. Tổng kết quá trình tiền xử lý

Quá trình tiền xử lý dữ liệu Crop Recommendation đã thực hiện các công việc sau:

1. Đọc và kiểm tra bộ dữ liệu ban đầu.
2. Kiểm tra giá trị thiếu và dữ liệu trùng lặp.
3. Kiểm tra tính hợp lệ của các đặc trưng.
4. Xác định 7 đặc trưng đầu vào và nhãn loại cây trồng.
5. Phân chia dữ liệu thành tập Train và Test theo tỷ lệ 80/20.
6. Lưu dữ liệu giữ nguyên thang đo và nhãn để phục vụ huấn luyện mô hình.

### Kết luận

Dữ liệu sau tiền xử lý được tổ chức thành các tập huấn luyện và kiểm tra riêng biệt.

Các tập dữ liệu này là đầu vào cho notebook 03. SoftMax, KNN và SVM chuẩn hóa dữ liệu trong Pipeline; các mô hình còn lại dùng dữ liệu gốc.

Kết quả tiền xử lý là đầu vào cho giai đoạn huấn luyện, so sánh và đánh giá các thuật toán học máy trong hệ thống khuyến nghị cây trồng.
